# MCP server

An **MCP server** lists tools and runs the tool a client requests. **MCP** means Model Context Protocol.

This notebook exposes two arXiv tools over **stdio**. The server is a child process. The client talks to it on standard input and output.

The tools match [direct tool calling](./mcp.ipynb). The runnable file is [`research_server.py`](./research_server.py).

## Related notes

- [Direct tool calling](./mcp.ipynb)
- [MCP client](./mcp-client.ipynb)
- [Reference servers](./reference-servers.ipynb)
- [Prompts and resources](./prompts-and-resources.ipynb)
- [Subprocess](./subprocess.ipynb)


## What the server handles

A tool server answers two requests.

- It lists its tools.
- It runs one tool.

There are two ways to build that server.

- **Low-level.** Your code handles `ListToolsRequest` and `CallToolRequest`.
- **FastMCP.** Each tool is a Python function. **FastMCP** builds the schema from the type hints and the docstring.

This notebook uses FastMCP.


## The research server

[`research_server.py`](./research_server.py) creates a FastMCP server named `research`.

`search_papers` queries arXiv. It writes `papers/<topic>/papers_info.json`. `extract_info` looks up one saved paper id.

The same file also defines resources and a prompt. Read those in [prompts and resources](./prompts-and-resources.ipynb).

The cell below is that tool portion, including the imports. The file then calls `mcp.run(transport="stdio")`. Run that from a terminal. A stdio server waits for a client.


In [ ]:
"""Research MCP server: arXiv tools, paper resources, and a search prompt."""

import json
import os
from typing import List

import arxiv
from mcp.server.fastmcp import FastMCP

PAPER_DIR = "papers"

mcp = FastMCP("research")


@mcp.tool()
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """Search for papers on arXiv based on a topic and store their information.

    Args:
        topic: The topic to search for.
        max_results: Maximum number of results to retrieve (default: 5).

    Returns:
        List of paper IDs found in the search.
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=topic,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance,
    )
    papers = client.results(search)

    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    file_path = os.path.join(path, "papers_info.json")

    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_id = paper.get_short_id()
        paper_ids.append(paper_id)
        papers_info[paper_id] = {
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "summary": paper.summary,
            "pdf_url": paper.pdf_url,
            "published": str(paper.published.date()),
        }

    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)

    print(f"Results are saved in: {file_path}")
    return paper_ids


@mcp.tool()
def extract_info(paper_id: str) -> str:
    """Search for information about a specific paper across all topic directories.

    Args:
        paper_id: The ID of the paper to look for.

    Returns:
        JSON string with paper information if found, otherwise an error message.
    """
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if not os.path.isdir(item_path):
            continue
        file_path = os.path.join(item_path, "papers_info.json")
        if not os.path.isfile(file_path):
            continue
        try:
            with open(file_path, "r") as json_file:
                papers_info = json.load(json_file)
                if paper_id in papers_info:
                    return json.dumps(papers_info[paper_id], indent=2)
        except (FileNotFoundError, json.JSONDecodeError) as error:
            print(f"Error reading {file_path}: {error}")
            continue

    return f"There's no saved information related to paper {paper_id}."


The file ends with `mcp.run(transport="stdio")`.

## Run the inspector

From `MCP/`:

```bash
npx @modelcontextprotocol/inspector uv run --with "mcp>=1.2,<2" --with arxiv research_server.py
```

In the inspector, set the command to `uv`. Set the arguments to:

`run --with mcp>=1.2,<2 --with arxiv research_server.py`

Pin `mcp` below version 2. The [28 July 2026 spec](https://blog.modelcontextprotocol.io/posts/2026-07-28/) changed `mcp.server.fastmcp`. This server uses the earlier interface.

Install [uv](https://docs.astral.sh/uv/getting-started/installation/) and [Node.js](https://docs.npmjs.com/downloading-and-installing-node-js-and-npm) first. The inspector is a Node package.

A warning about the field `lifespan` can come from `pydantic-settings`. The server still starts.

Saved papers appear under [`papers/`](./papers).


## Further reading

- [Low-level server example](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/servers/simple-tool/mcp_simple_tool/server.py)
- [Low-level advanced usage](https://github.com/modelcontextprotocol/python-sdk/blob/main/README.md#advanced-usage)
- [MCP inspector](https://github.com/modelcontextprotocol/inspector)
- [FastMCP](https://github.com/jlowin/fastmcp)
- [Server quickstart](https://modelcontextprotocol.io/quickstart/server)
